# 04 — Matrix Multiplication Kernels

Covers Phase 4 of the kernel roadmap:
- `naive_matmul` — no tiling, baseline
- `tiled_matmul` — SRAM tiling + autotuning
- `batched_matmul` — BMM with batch dimension

**Metric**: TFLOPS — `(2 × M × N × K × 1e-12) / (ms × 1e-3)`

In [ ]:
# ── Setup: mount Drive and clone / pull the repo ─────────────────────────────
import os
from google.colab import drive

drive.mount("/content/drive")

REPO_URL    = "https://github.com/Bhavikupadhyay/triton-kernels.git"
REPO_BRANCH = "main"
REPO_DIR    = "/content/drive/MyDrive/triton-kernels"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch --all
    !git -C {REPO_DIR} checkout -f {REPO_BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git rev-parse --abbrev-ref HEAD
!bash scripts/setup_colab.sh

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import torch
import triton

from kernels.matmul.naive_matmul import naive_matmul, test_naive_matmul, benchmark_naive_matmul
from kernels.matmul.tiled_matmul import tiled_matmul, test_tiled_matmul, benchmark_tiled_matmul
from kernels.matmul.batched_matmul import (
    batched_matmul, test_batched_matmul,
    benchmark_batched_matmul, benchmark_batched_matmul_steady,
)

print("Imports ready")

## 1. naive_matmul

**File**: `kernels/matmul/naive_matmul.py`  
**PyTorch equivalent**: `torch.matmul(A, B)`  
**Key**: No SRAM tiling — serves as a baseline to show why tiling matters.

In [ ]:
# ── naive_matmul: Correctness ─────────────────────────────────────────────────
test_naive_matmul()

In [ ]:
# ── naive_matmul: Benchmark ───────────────────────────────────────────────────
import os
os.makedirs("benchmarks/results/matmul", exist_ok=True)

benchmark_naive_matmul.run(
    print_data=True,
    show_plots=True,
    save_path="benchmarks/results/matmul",
)

**Interpretation — naive_matmul (T4, fp32, square M=N=K)**

**cuBLAS wins at every size except N=256, and the gap is 2–4×.** This is the expected result — naive exists to show the baseline before tiling is added.

**N=256 — Triton ahead (0.75 vs 0.62 TFLOPS).** Both are latency-dominated; the matrix is too small to saturate the GPU. Triton's launch overhead is lower here, giving it a slight edge.

**N=512 — cuBLAS jumps to 4.37 TFLOPS; Triton reaches 1.06.** This is where cuBLAS's SRAM tiling begins to pay off. It loads tiles of A and B into shared memory and reuses them across the dot products for that tile, keeping HBM traffic proportional to the tile size rather than the full row/column. Our kernel loads fresh data from HBM on every K-step with no reuse.

**N=1024 — Triton peaks at 2.48 TFLOPS.** The K-loop is long enough that each program does meaningful compute per HBM load, pushing utilization up. cuBLAS peaks here too at 5.60 TFLOPS (~69% of T4's 8.1 TFLOPS fp32 theoretical).

**N≥2048 — both drop; Triton more sharply.** Once the full A and B matrices exceed L2 (4 MB on T4 — a 1024×1024 fp32 matrix is exactly 4 MB), every load in the K-loop is a cold HBM read. Without group ordering, adjacent programs in the launch grid share no A or B tiles in L2, so each program independently re-reads its entire row and column stripe from HBM. Triton drops to 1.93 and 1.77 TFLOPS. cuBLAS drops too but less severely because its SRAM tiling limits how much data each block needs to pull from HBM per output tile.

**Absolute numbers are low for both.** T4 fp32 theoretical is 8.1 TFLOPS (Turing tensor cores only support FP16/INT8 at high throughput, not FP32). Triton peaks at 31% of theoretical; cuBLAS at 69%. Neither is close to the ceiling — matmul efficiency is dominated by how well you tile for SRAM reuse, which is exactly what tiled_matmul addresses.

## 2. tiled_matmul

**File**: `kernels/matmul/tiled_matmul.py`  
**Key**: SRAM tiling reduces HBM bandwidth. Uses `@triton.autotune` to find optimal `BLOCK_M`, `BLOCK_N`, `BLOCK_K`.

In [ ]:
# ── tiled_matmul: Correctness ─────────────────────────────────────────────────
test_tiled_matmul()

In [ ]:
# ── tiled_matmul: Benchmark ───────────────────────────────────────────────────
import os
os.makedirs("benchmarks/results/matmul", exist_ok=True)

benchmark_tiled_matmul.run(
    print_data=True,
    show_plots=True,
    save_path="benchmarks/results/matmul",
)

**Interpretation — tiled_matmul (T4, fp32, square M=N=K)**

**Triton matches cuBLAS at N=4096 (3.84 vs 3.89 TFLOPS — within noise) and wins at N=256.** PyTorch is ahead at N=512–1024 by 16–25%, but the gap closes to 4% at N=2048 and essentially disappears at N=4096.

**N=256 — Triton ahead by 10% (1.48 vs 1.35 TFLOPS).** Small matrix, latency dominated. Same pattern as naive.

**N=512–1024 — PyTorch ahead by 16–25%.** The autotuner's winning config at these sizes likely uses BLOCK_M=128, which requires enough output tiles to keep all SMs busy. At N=512 with BLOCK_M=128 there are only 16 tile rows — too few to saturate T4's 40 SMs fully. cuBLAS carries size-specific heuristics that pick smaller blocks for small matrices. Our config set doesn't include anything below BLOCK_M=32 with appropriate staging for small N.

**N=2048–4096 — gap closes to 4% then ~1%.** Large matrices produce enough tiles that BLOCK_M=128 (or 64) keeps every SM busy, and group ordering ensures A tiles get L2 reuse across GROUP_SIZE programs. At N=4096 both reach ~3.84–3.89 TFLOPS.

**Compare to naive at large N.** Tiling improves Triton by 2.2× at N=2048 (4.19 vs 1.93) and 2.2× at N=4096 (3.84 vs 1.77). The drop-then-plateau that naive showed is gone — tiled stays flat across N=1024–4096.

**Both plateau around 4–5 TFLOPS — well below T4's 8.1 TFLOPS fp32 theoretical.** T4's tensor cores only support FP16/INT8 at high throughput. fp32 computation goes through CUDA cores with no tensor core acceleration, and cuBLAS hits 60% of that ceiling. Tiled Triton hits 52%. To go further requires fp16 inputs so tensor cores engage — which is what the attention kernels in Phase 6 will use.

## 3. batched_matmul

**File**: `kernels/matmul/batched_matmul.py`  
**PyTorch equivalent**: `torch.bmm(A, B)`  
**Key**: Adds a batch dimension using `tl.program_id(2)`.

In [ ]:
# ── batched_matmul: Correctness ───────────────────────────────────────────────
test_batched_matmul()

In [ ]:
# ── batched_matmul: Benchmark (first run — autotuner warms the GPU) ───────────
import os
os.makedirs("benchmarks/results/matmul", exist_ok=True)

benchmark_batched_matmul.run(
    print_data=True,
    show_plots=True,
    save_path="benchmarks/results/matmul",
)

# ── batched_matmul: Benchmark (steady state — autotuner cache is hot) ─────────
benchmark_batched_matmul_steady.run(
    print_data=True,
    show_plots=True,
    save_path="benchmarks/results/matmul",
)

**Interpretation — batched_matmul (T4, fp32, B=16, square M=N=K)**

**At N≥1024, Triton and PyTorch bmm are within 5% of each other; Triton pulls ahead by ~10% at N=4096.** Small-N results are dominated by GPU clock state and are not meaningful for comparison.

**N=256–512 — unstable across runs.** N=256 Triton ranges from 1.78 to 4.44 TFLOPS depending on when in the session the benchmark runs. The matrix is small enough that each kernel call takes ~0.1ms; `warmup=25` iterations (≈2.5ms total) is insufficient to drive the GPU from idle into sustained boost clock. PyTorch varies too (3.98–5.09). Nothing can be concluded from these numbers.

**N=1024–2048 — within noise of tiled_matmul.** Triton and PyTorch both land near 3.6–4.0 TFLOPS, nearly identical to the 2D case. The batch dimension (B=16) adds 16× the tiles to the launch grid rather than introducing any new memory bottleneck — each batch element's matrix is still the same size, and group ordering within each element works identically.

**N=4096 — Triton ahead by ~10–13% (3.35–3.47 vs 2.97–3.09) across all runs.** This matches the tiled_matmul result at the same size. The batch dimension doesn't meaningfully change the large-N behaviour because at N=4096 the bottleneck is HBM bandwidth and compute per tile, both of which are identical across 2D and batched versions.

**First-run vs steady-state: what the two benchmarks actually show.**
The `batched_matmul` benchmark fires the autotuner on its first call, which compiles and measures all eight configs before `do_bench` runs. That compilation work drives the GPU into sustained boost clock, inflating the first measurement (most visible at N=256). `batched_matmul_steady` runs immediately after with a hot autotuner cache and a warm GPU — numbers are closer to steady-state. On a second cell execution, the autotuner cache is already populated so `batched_matmul` no longer pre-warms, and the N=256 result drops sharply (1.78 TFLOPS). The effect disappears above N=1024 because larger kernels are long enough that `warmup=25` iterations are sufficient to stabilise clock speed regardless of prior state.

In [ ]:
# ── Summary Table ────────────────────────────────────────────────────────────
# import pandas as pd, glob
# csvs = glob.glob("benchmarks/results/matmul/*.csv")
# if csvs:
#     print(pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True).to_string(index=False))
# else:
#     print("No CSVs yet.")